# Tiền xử lý dữ liệu dùng chung (NLBSE'23 Code Comment Classification)

Notebook này tạo **một bộ dữ liệu chung** cho cả 3 ngôn ngữ (Python, Java, Pharo). Mọi mô hình (TF-IDF + ML, RoBERTa, ...) đều đọc cùng một bộ train/test/folds nên kết quả so sánh được với nhau.

Notebook **không** trích đặc trưng hay tokenize. TF-IDF, tokenizer, class weights... thuộc về từng notebook huấn luyện; riêng TF-IDF phải được fit trên phần train của từng fold để không rò rỉ từ vựng/IDF sang validation.

## Quy trình
1. **Kiểm tra raw**: dữ liệu thiếu, nhãn sai, ID trùng hoặc thông tin mâu thuẫn.
2. **Chuyển sang multi-label**: mỗi câu một dòng, mỗi nhãn một cột 0/1.
3. **Chuẩn hóa nhẹ**: khoảng trắng, xuống dòng, tab; giữ chữ hoa/thường và ký hiệu code.
4. **Nhóm các câu trùng nội dung** (`group_id`): để chúng không bị chia sang cả train và validation/test.
5. **Chia train/test và 5-fold trên train**: cân bằng nhãn, giữ các nhóm câu cùng phía, cố định seed.
6. **Lưu và kiểm tra** bộ dữ liệu chung.

## Output
```
data/processed/
├── python/
│   ├── train.csv        # comment_sentence_id, class, group_id, text, các nhãn
│   ├── test.csv
│   ├── folds.json       # chỉ số train/validation (vị trí dòng, bắt đầu từ 0) trong train.csv
│   └── label_cols.json  # thứ tự nhãn
├── java/
└── pharo/
```

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path('../data')
OUT_DIR = RAW_DIR / 'processed'

# Nhãn chuẩn của NLBSE'23 cho từng ngôn ngữ. Danh sách này cũng là thứ tự cột nhãn trong output.
LABEL_COLS = {
    'python': ['DevelopmentNotes', 'Expand', 'Parameters', 'Summary', 'Usage'],
    'java':   ['deprecation', 'Expand', 'Ownership', 'Pointer', 'rational', 'summary', 'usage'],
    'pharo':  ['Classreferences', 'Collaborators', 'Example', 'Intent',
               'Keyimplementationpoints', 'Keymessages', 'Responsibilities'],
}
LANGS = list(LABEL_COLS)

SEED         = 42
N_TEST_PARTS = 5   # chia dữ liệu thành 5 phần và lấy 1 phần làm test (≈ 20%)
N_FOLDS      = 5   # số fold CV trên train

## Bước 1: Kiểm tra raw
File raw ở dạng long: mỗi câu có một dòng cho từng category, `instance_type` = 1 nếu câu thuộc category đó.

- **Lỗi** (phải bằng 0, nếu không notebook dừng): thiếu/rỗng `class` hoặc text, `instance_type` không phải 0/1, category lạ, cặp (ID, category) bị trùng, một ID có nhiều text/class, ID thiếu category.
- **Thông tin**: cột `partition` của raw là phân vùng train/test *theo từng category* nên cùng một câu có thể rơi vào cả hai phía. Cột này không dùng, vì ta tự chia ở Bước 5.

In [2]:
def audit_raw(df, labels):
    '''Trả về (lỗi, thông tin) cho một file raw dạng long.'''
    by_id = df.groupby('comment_sentence_id')
    text_cols = df[['class', 'comment_sentence']]
    blank = text_cols.isna() | text_cols.astype(str).apply(lambda s: s.str.strip() == '')
    errors = {
        'dòng thiếu/rỗng class hoặc text':      int(blank.any(axis=1).sum()),
        'instance_type không phải 0/1':         int((~df['instance_type'].isin([0, 1])).sum()),
        'category ngoài danh sách nhãn chuẩn':  int((~df['category'].isin(labels)).sum()),
        'cặp (ID, category) bị trùng':          int(df.duplicated(['comment_sentence_id', 'category']).sum()),
        'ID có >1 text hoặc >1 class':          int(((by_id['comment_sentence'].nunique() > 1)
                                                     | (by_id['class'].nunique() > 1)).sum()),
        'ID thiếu category':                    int((by_id['category'].nunique() < len(labels)).sum()),
    }
    info = {
        'số dòng (dạng long)':                        len(df),
        'số ID (câu)':                                df['comment_sentence_id'].nunique(),
        'ID có partition khác nhau giữa category':    int((by_id['partition'].nunique() > 1).sum()),
    }
    return errors, info


# keep_default_na=False: không để pandas biến các câu như 'none', 'null', 'na' thành NaN
raw = {lang: pd.read_csv(RAW_DIR / f'{lang}.csv', keep_default_na=False) for lang in LANGS}
reports = {lang: audit_raw(raw[lang], LABEL_COLS[lang]) for lang in LANGS}

print(pd.DataFrame({lang: {**errors, **info} for lang, (errors, info) in reports.items()}).to_string())
for lang, (errors, _) in reports.items():
    assert not any(errors.values()), f'{lang}: raw có lỗi {errors}'
print('\nRaw hợp lệ: các mục lỗi đều bằng 0.')

                                         python   java  pharo
dòng thiếu/rỗng class hoặc text               0      0      0
instance_type không phải 0/1                  0      0      0
category ngoài danh sách nhãn chuẩn           0      0      0
cặp (ID, category) bị trùng                   0      0      0
ID có >1 text hoặc >1 class                   0      0      0
ID thiếu category                             0      0      0
số dòng (dạng long)                       12775  16926  12355
số ID (câu)                                2555   2418   1765
ID có partition khác nhau giữa category    1231   1229    931

Raw hợp lệ: các mục lỗi đều bằng 0.


## Bước 2: Chuyển sang multi-label
Pivot từ dạng long sang dạng wide: mỗi `comment_sentence_id` một dòng, mỗi nhãn một cột 0/1. Số positive của từng nhãn được đối soát lại với raw.

In [3]:
def to_multilabel(df, labels):
    '''Dạng long -> wide: comment_sentence_id | class | text | các nhãn 0/1.'''
    y = df.pivot(index='comment_sentence_id', columns='category', values='instance_type')[labels]
    meta = df.drop_duplicates('comment_sentence_id').set_index('comment_sentence_id')[['class', 'comment_sentence']]
    wide = meta.join(y).rename(columns={'comment_sentence': 'text'})
    return wide.reset_index().sort_values('comment_sentence_id', ignore_index=True)


wide = {lang: to_multilabel(raw[lang], LABEL_COLS[lang]) for lang in LANGS}

for lang in LANGS:
    labels, df = LABEL_COLS[lang], wide[lang]
    positives_raw = raw[lang].groupby('category')['instance_type'].sum()[labels]
    assert (df[labels].sum() == positives_raw).all(), f'{lang}: số positive không khớp raw'
    assert df['comment_sentence_id'].is_unique
    assert (df[labels].sum(axis=1) > 0).all(), f'{lang}: có câu không có nhãn nào'

    print(f'== {lang}: {len(df):,} câu, trung bình {df[labels].sum(axis=1).mean():.2f} nhãn/câu')
    print(pd.DataFrame({'positive': df[labels].sum(), '%': (df[labels].mean() * 100).round(2)}).to_string())
    print()

== python: 2,555 câu, trung bình 1.12 nhãn/câu
                  positive      %
DevelopmentNotes       312  12.21
Expand                 504  19.73
Parameters             794  31.08
Summary                454  17.77
Usage                  800  31.31

== java: 2,418 câu, trung bình 1.18 nhãn/câu
             positive      %
deprecation       127   5.25
Expand            632  26.14
Ownership         115   4.76
Pointer           364  15.05
rational          280  11.58
summary           415  17.16
usage             912  37.72

== pharo: 1,765 câu, trung bình 1.16 nhãn/câu
                         positive      %
Classreferences                77   4.36
Collaborators                 127   7.20
Example                       748  42.38
Intent                        218  12.35
Keyimplementationpoints       232  13.14
Keymessages                   305  17.28
Responsibilities              336  19.04



## Bước 3: Chuẩn hóa nhẹ
Chỉ gộp mọi chuỗi khoảng trắng (dấu cách, tab, xuống dòng) thành một dấu cách và cắt hai đầu. **Giữ nguyên** chữ hoa/thường và ký hiệu code (`@param`, `#method`, `snake_case`, ...) vì các mô hình sau đó (char n-gram, tokenizer BPE) cần chúng.

In [4]:
def normalize_text(text):
    return re.sub(r'\s+', ' ', text).strip()


for lang in LANGS:
    df = wide[lang]
    normalized = df['text'].map(normalize_text)
    print(f'{lang:<7}: {int((normalized != df["text"]).sum())} câu thay đổi sau chuẩn hóa')
    df['text'] = normalized
    assert (df['text'] != '').all(), f'{lang}: có câu rỗng sau chuẩn hóa'

python : 0 câu thay đổi sau chuẩn hóa
java   : 0 câu thay đổi sau chuẩn hóa
pharo  : 0 câu thay đổi sau chuẩn hóa


## Bước 4: Nhóm các câu trùng nội dung
Các câu có text giống hệt nhau (sau chuẩn hóa) nhận cùng một `group_id`. Ở Bước 5 mỗi nhóm được giữ nguyên một phía, nên một câu trong validation/test không có bản sao nằm trong train.

Các nhóm trùng text nhưng khác nhãn được báo cáo để biết; ta không xóa vì không có căn cứ chọn nhãn nào đúng.

In [5]:
rows = {}
for lang in LANGS:
    df, labels = wide[lang], LABEL_COLS[lang]
    df['group_id'] = pd.factorize(df['text'])[0]      # đánh số theo thứ tự ID, nên ổn định giữa các lần chạy

    sizes = df['group_id'].value_counts()
    members = df[df['group_id'].isin(sizes[sizes > 1].index)]
    rows[lang] = {
        'số câu':                              len(df),
        'số nhóm (group_id)':                  len(sizes),
        'nhóm có >1 câu':                      int((sizes > 1).sum()),
        'số câu nằm trong nhóm >1 câu':        len(members),
        'kích thước nhóm lớn nhất':            int(sizes.max()),
        'nhóm trùng text nhưng khác nhãn':     int(members.groupby('group_id')[labels].nunique().gt(1).any(axis=1).sum()),
        'nhóm trùng text ở >1 class':          int(members.groupby('group_id')['class'].nunique().gt(1).sum()),
    }
    wide[lang] = df[['comment_sentence_id', 'class', 'group_id', 'text'] + labels]

print(pd.DataFrame(rows).to_string())

                                 python  java  pharo
số câu                             2555  2418   1765
số nhóm (group_id)                 2299  1668   1620
nhóm có >1 câu                      112   252     60
số câu nằm trong nhóm >1 câu        368  1002    205
kích thước nhóm lớn nhất             19    51     18
nhóm trùng text nhưng khác nhãn      16    13      5
nhóm trùng text ở >1 class           80    56     34


## Bước 5: Chia train/test và 5-fold trên train
Cần đồng thời: (a) tỷ lệ từng nhãn giống nhau giữa các phần, (b) mọi câu cùng `group_id` ở cùng một phía. `MultilabelStratifiedKFold` làm được (a) nhưng không biết về nhóm, nên ở đây dùng bộ chia tham lam theo nhóm:

- Gộp nhãn theo nhóm: mỗi nhóm có vector số positive từng nhãn và số câu.
- Duyệt các nhóm từ lớn đến nhỏ (thứ tự ngẫu nhiên có seed khi bằng nhau), gán mỗi nhóm vào phần làm tổng độ lệch tương đối so với mức lý tưởng tăng ít nhất.
- Test là phần cân bằng nhãn nhất trong 5 phần (≈ 20%); 5 fold CV được chia lại từ train bằng cùng thuật toán.

Một nhóm rất lớn có nhãn hiếm (ví dụ 32 câu `@author kevin bourrillion` của nhãn `Ownership` trong Java) không thể tách ra, nên nhãn đó sẽ lệch hơn các nhãn còn lại.

In [6]:
def group_multilabel_kfold(Y, groups, n_splits, seed):
    '''Chia thành n_splits phần, cân bằng nhãn + số câu, mỗi nhóm nằm trọn trong một phần.
    Trả về list các mảng chỉ số dòng (đã sắp xếp), mỗi phần một mảng.'''
    _, group_of_row = np.unique(groups, return_inverse=True)
    n_groups = group_of_row.max() + 1
    group_labels = np.zeros((n_groups, Y.shape[1]))
    np.add.at(group_labels, group_of_row, Y)
    group_size = np.bincount(group_of_row).astype(float)

    feats = np.column_stack([group_labels, group_size])   # mỗi nhóm: [số positive từng nhãn..., số câu]
    target = feats.sum(axis=0) / n_splits                 # mức lý tưởng cho mỗi phần
    rng = np.random.default_rng(seed)

    order = rng.permutation(n_groups)
    order = order[np.argsort(-group_size[order], kind='stable')]   # nhóm lớn trước, hòa thì ngẫu nhiên

    part_feats = np.zeros((n_splits, feats.shape[1]))
    part_of_group = np.empty(n_groups, dtype=int)
    for g in order:
        dev_before = (part_feats - target) / target
        dev_after  = (part_feats + feats[g] - target) / target
        cost = (dev_after ** 2 - dev_before ** 2).sum(axis=1)      # độ tăng tổng độ lệch nếu g vào từng phần
        part = rng.choice(np.flatnonzero(cost <= cost.min() + 1e-12))
        part_of_group[g] = part
        part_feats[part] += feats[g]

    part_of_row = part_of_group[group_of_row]
    return [np.flatnonzero(part_of_row == p) for p in range(n_splits)]


def make_split(df, labels):
    '''Trả về train, test (DataFrame, index 0..n-1) và folds (list dict) với chỉ số theo dòng của train.'''
    Y = df[labels].to_numpy()
    parts = group_multilabel_kfold(Y, df['group_id'].to_numpy(), N_TEST_PARTS, SEED)

    ideal = Y.sum(axis=0) / N_TEST_PARTS
    imbalance = [(((Y[p].sum(axis=0) / ideal) - 1) ** 2).sum() for p in parts]
    test_part = int(np.argmin(imbalance))

    test_idx = parts[test_part]
    train_idx = np.sort(np.concatenate([p for i, p in enumerate(parts) if i != test_part]))
    train = df.iloc[train_idx].reset_index(drop=True)
    test = df.iloc[test_idx].reset_index(drop=True)

    cv_parts = group_multilabel_kfold(train[labels].to_numpy(), train['group_id'].to_numpy(), N_FOLDS, SEED)
    all_rows = np.arange(len(train))
    folds = [{'fold': k,
              'train_indices': np.setdiff1d(all_rows, val).tolist(),
              'val_indices': val.tolist()} for k, val in enumerate(cv_parts)]
    return train, test, folds


splits = {lang: make_split(wide[lang], LABEL_COLS[lang]) for lang in LANGS}

for lang in LANGS:
    labels, (train, test, folds) = LABEL_COLS[lang], splits[lang]
    n = len(wide[lang])
    print(f'== {lang}: train {len(train):,} ({len(train) / n:.1%}) | test {len(test):,} ({len(test) / n:.1%})'
          f' | val các fold {[len(f["val_indices"]) for f in folds]}')

    balance = pd.DataFrame({
        'all (%)':   wide[lang][labels].mean() * 100,
        'train (%)': train[labels].mean() * 100,
        'test (%)':  test[labels].mean() * 100,
    })
    balance['test/all'] = balance['test (%)'] / balance['all (%)']

    y_train = train[labels].to_numpy()
    val_pos = np.array([y_train[f['val_indices']].sum(axis=0) for f in folds])
    ratio = val_pos / (y_train.sum(axis=0) / N_FOLDS)
    balance['val/ideal min'] = ratio.min(axis=0)
    balance['val/ideal max'] = ratio.max(axis=0)
    print(balance.round(3).to_string())
    print()

== python: train 2,044 (80.0%) | test 511 (20.0%) | val các fold [409, 409, 408, 409, 409]
                  all (%)  train (%)  test (%)  test/all  val/ideal min  val/ideal max
DevelopmentNotes   12.211     12.231    12.133     0.994          1.000          1.000
Expand             19.726     19.716    19.765     1.002          0.993          1.005
Parameters         31.076     31.067    31.115     1.001          1.000          1.000
Summary            17.769     17.759    17.808     1.002          0.992          1.006
Usage              31.311     31.311    31.311     1.000          0.992          1.008

== java: train 1,934 (80.0%) | test 484 (20.0%) | val các fold [386, 386, 388, 387, 387]
             all (%)  train (%)  test (%)  test/all  val/ideal min  val/ideal max
deprecation    5.252      5.274     5.165     0.983          0.980          1.029
Expand        26.137     26.163    26.033     0.996          0.998          1.008
Ownership      4.756      4.860     4.339     0.912

## Bước 6: Lưu và kiểm tra
Sau khi ghi, đọc lại từ đĩa (bằng `pd.read_csv` mặc định, giống cách các notebook huấn luyện sẽ đọc) và kiểm tra:

- cột, kiểu dữ liệu, không thiếu giá trị, nhãn chỉ gồm 0/1; ghi ra rồi đọc lại không đổi;
- ID không trùng, train ∪ test đúng bằng tập ID raw, train ∩ test rỗng;
- không `group_id` nào xuất hiện ở cả train và test, và `group_id` ↔ text là tương ứng 1-1;
- tổng positive từng nhãn (train + test) khớp raw;
- `folds.json`: mỗi fold có train/val rời nhau và phủ đủ train, mỗi dòng của train nằm trong validation đúng một lần, không nhóm nào nằm ở cả train và val của cùng một fold;
- chạy lại thuật toán chia cho kết quả giống hệt (seed cố định).

In [7]:
def save_split(lang, train, test, folds, labels):
    out = OUT_DIR / lang
    out.mkdir(parents=True, exist_ok=True)
    train.to_csv(out / 'train.csv', index=False)
    test.to_csv(out / 'test.csv', index=False)
    (out / 'folds.json').write_text(json.dumps(folds), encoding='utf-8')
    (out / 'label_cols.json').write_text(json.dumps(labels, indent=2), encoding='utf-8')


def validate(lang, raw_df, wide_df, train_expected, test_expected, folds_expected):
    out, labels = OUT_DIR / lang, LABEL_COLS[lang]
    train, test = pd.read_csv(out / 'train.csv'), pd.read_csv(out / 'test.csv')
    folds = json.loads((out / 'folds.json').read_text(encoding='utf-8'))
    saved_labels = json.loads((out / 'label_cols.json').read_text(encoding='utf-8'))

    # cột, giá trị, round-trip
    assert saved_labels == labels
    for name, d in [('train', train), ('test', test)]:
        assert list(d.columns) == ['comment_sentence_id', 'class', 'group_id', 'text'] + labels, f'{lang}/{name}: sai cột'
        assert not d.isna().any().any(), f'{lang}/{name}: có giá trị thiếu'
        assert (d['text'].str.strip() == d['text']).all() and (d['text'] != '').all()
        assert d[labels].isin([0, 1]).all().all()
        assert d['comment_sentence_id'].is_unique
    pd.testing.assert_frame_equal(train, train_expected)
    pd.testing.assert_frame_equal(test, test_expected)

    # ID và nhóm
    assert not set(train['comment_sentence_id']) & set(test['comment_sentence_id']), f'{lang}: ID rò rỉ train/test'
    assert set(train['comment_sentence_id']) | set(test['comment_sentence_id']) == set(raw_df['comment_sentence_id'])
    assert not set(train['group_id']) & set(test['group_id']), f'{lang}: group_id rò rỉ train/test'
    both = pd.concat([train, test])
    assert both.groupby('group_id')['text'].nunique().eq(1).all()
    assert both.groupby('text')['group_id'].nunique().eq(1).all()

    # nhãn khớp raw
    assert (both[labels].sum() == raw_df.groupby('category')['instance_type'].sum()[labels]).all()

    # folds
    assert len(folds) == N_FOLDS
    n_train, groups = len(train), train['group_id'].to_numpy()
    times_in_val = np.zeros(n_train, dtype=int)
    for k, fold in enumerate(folds):
        tr, va = np.array(fold['train_indices']), np.array(fold['val_indices'])
        assert fold['fold'] == k
        assert np.array_equal(np.sort(np.concatenate([tr, va])), np.arange(n_train)), f'{lang} fold {k}: không phủ đủ train'
        assert not set(groups[tr]) & set(groups[va]), f'{lang} fold {k}: nhóm nằm ở cả train và val'
        times_in_val[va] += 1
    assert (times_in_val == 1).all()

    # tái lập
    train2, test2, folds2 = make_split(wide_df, labels)
    assert train2.equals(train_expected) and test2.equals(test_expected) and folds2 == folds, f'{lang}: không tái lập được'

    return {'train': len(train), 'test': len(test), 'nhãn': len(labels), 'nhóm trong train': train['group_id'].nunique(),
            'val các fold': [len(f['val_indices']) for f in folds]}


summary = {}
for lang in LANGS:
    train, test, folds = splits[lang]
    save_split(lang, train, test, folds, LABEL_COLS[lang])
    summary[lang] = validate(lang, raw[lang], wide[lang], train, test, folds)

print(pd.DataFrame(summary).T.to_string())
print('\nTất cả kiểm tra đều đạt. Các file trong', OUT_DIR.resolve())
for path in sorted(OUT_DIR.glob('*/*')):
    print(f'  {path.relative_to(OUT_DIR)!s:<28} {path.stat().st_size / 1024:>8.1f} KB')

       train test nhãn nhóm trong train               val các fold
python  2044  511    5             1838  [409, 409, 408, 409, 409]
java    1934  484    7             1326  [386, 386, 388, 387, 387]
pharo   1411  354    7             1292  [283, 282, 282, 282, 282]

Tất cả kiểm tra đều đạt. Các file trong /home/linhntn16/projects/introDS/data/processed
  java/folds.json                  51.5 KB
  java/label_cols.json              0.1 KB
  java/test.csv                    44.0 KB
  java/train.csv                  174.6 KB
  pharo/folds.json                 36.2 KB
  pharo/label_cols.json             0.1 KB
  pharo/test.csv                   32.5 KB
  pharo/train.csv                 134.3 KB
  python/folds.json                54.7 KB
  python/label_cols.json            0.1 KB
  python/test.csv                  36.0 KB
  python/train.csv                143.1 KB
